# Data Governance — Maven Market UC

> **Catalog:** `maven_market_uc` &nbsp;|&nbsp; **Schema:** `gold` &nbsp;|&nbsp; **Engine:** Databricks SQL &nbsp;|&nbsp; **Policy Enforcement:** Dynamic (`is_member()`)

---

## Overview

This notebook establishes a **fine-grained security layer** over the Maven Market Unity Catalog data warehouse. It enforces **who can see what** at both the row and column level, ensuring every query returns only the data the caller is authorized to access — without requiring separate physical copies of the data.

---

## Security Controls Implemented

| Control | Technique | Scope |
|---|---|---|
| **Row-Level Security (RLS)** | `WHERE` clause + `is_member()` | `fact_sales`, `fact_returns` |
| **Column-Level Security (CLS)** | `CASE` expression + `is_member()` | `fact_sales`, `dim_customer`, `dim_product` |
| **Object-Level Privileges** | `GRANT SELECT` | Aggregate tables → `executives` group |

---

## Secure Views Created

### 1. `v_secure_fact_sales` — RLS + CLS
| Layer | Rule |
|---|---|
| **RLS** | `admins`, `data_engineers`, `data_analysts`, `business_users` → **all regions** |
| | `central_managers` → **Central region only** |
| | `other_region_managers` → **all regions except Central** |
| **CLS** | `total_revenue`, `total_cost`, `total_profit` visible only to `admins` and `data_engineers`; masked as `NULL` for everyone else |

### 2. `v_secure_fact_returns` — RLS
| Layer | Rule |
|---|---|
| **RLS** | Same region-based filtering as sales (Central vs. non-Central managers) |

### 3. `v_secure_dim_customer` — CLS
| Column | Visibility |
|---|---|
| `customer_name` | `admins` only; otherwise **REDACTED** |
| `yearly_income` | `admins` and `data_engineers`; otherwise **RESTRICTED** |
| All other columns | Visible to all authenticated users |

### 4. `v_secure_dim_product` — CLS
| Column | Visibility |
|---|---|
| `product_cost` | `admins` and `data_engineers`; otherwise `NULL` |
| All other columns | Visible to all authenticated users |

---

## Executive Access Grants

The `executives` group receives **read-only access** to pre-aggregated, non-sensitive tables:

* `agg_daily_sales`
* `agg_monthly_sales`
* `agg_store_performance`
* `agg_return_kpi`

This ensures executives can self-serve on KPIs without ever touching row-level transactional data.

---

## Group Privilege Matrix

| Group | Sales Rows | Returns Rows | Financial Cols | PII Cols |
|---|---|---|---|---|
| `admins` | All | All | Visible | Visible |
| `data_engineers` | All | All | Visible | Partial |
| `data_analysts` | All | All | Masked | Masked |
| `business_users` | All | All | Masked | Masked | 
| `central_managers` | Central only | Central only | Masked | Masked |
| `other_region_managers` | Non-Central | Non-Central | Masked | Masked |

##SECURE SALES (RLS + CLS)

In [0]:

CREATE OR REPLACE VIEW maven_market_uc.gold.v_secure_fact_sales AS
SELECT
    transaction_date,
    f.store_id,
    product_id,

    -- region from dim_store
    s.sales_region,

    quantity,

    -- CLS (mask financials)
    CASE 
        WHEN is_member('admins') OR is_member('data_engineers')
        THEN total_revenue ELSE NULL END AS total_revenue,

    CASE 
        WHEN is_member('admins') OR is_member('data_engineers')
        THEN total_cost ELSE NULL END AS total_cost,

    CASE 
        WHEN is_member('admins') OR is_member('data_engineers')
        THEN total_profit ELSE NULL END AS total_profit

FROM maven_market_uc.gold.fact_sales f
LEFT JOIN maven_market_uc.gold.dim_store s
ON f.store_id = s.store_id

WHERE
    is_member('admins')
    OR is_member('data_engineers')
    OR is_member('data_analysts')
    OR is_member('business_users')
    OR (is_member('central_managers') AND s.sales_region = 'Central')
    OR (is_member('other_region_managers') AND s.sales_region != 'Central')

##SECURE RETURNS (RLS)

In [0]:

CREATE OR REPLACE VIEW maven_market_uc.gold.v_secure_fact_returns AS
SELECT
    r.*,
    s.sales_region
FROM maven_market_uc.gold.fact_returns r
LEFT JOIN maven_market_uc.gold.dim_store s
ON r.store_id = s.store_id

WHERE
    is_member('admins')
    OR is_member('data_engineers')
    OR is_member('data_analysts')
    OR is_member('business_users')
    OR (is_member('central_managers') AND s.sales_region = 'Central')
    OR (is_member('other_region_managers') AND s.sales_region != 'Central')

##SECURE CUSTOMER (CLS)

In [0]:

CREATE OR REPLACE VIEW maven_market_uc.gold.v_secure_dim_customer AS
SELECT
    customer_id,

    -- mask name
    CASE 
        WHEN is_member('admins') THEN customer_name
        ELSE 'REDACTED'
    END AS customer_name,

    customer_city,
    customer_state_province,
    customer_country,

    gender,
    marital_status,

    -- mask income
    CASE 
        WHEN is_member('admins') OR is_member('data_engineers')
        THEN yearly_income
        ELSE 'RESTRICTED'
    END AS yearly_income,

    education,
    occupation,
    homeowner

FROM maven_market_uc.gold.dim_customer

##SECURE PRODUCT (CLS)

In [0]:

CREATE OR REPLACE VIEW maven_market_uc.gold.v_secure_dim_product AS
SELECT
    product_id,
    product_name,
    product_brand,
    product_retail_price,

    CASE 
        WHEN is_member('admins') OR is_member('data_engineers')
        THEN product_cost
        ELSE NULL
    END AS product_cost,

    product_weight,
    recyclable,
    low_fat

FROM maven_market_uc.gold.dim_product

##AGG TABLES (EXECUTIVE SAFE)

In [0]:

GRANT SELECT ON maven_market_uc.gold.agg_daily_sales TO `executives`;
GRANT SELECT ON maven_market_uc.gold.agg_monthly_sales TO `executives`;
GRANT SELECT ON maven_market_uc.gold.agg_store_performance TO `executives`;
GRANT SELECT ON maven_market_uc.gold.agg_return_kpi TO `executives`;